In [2]:
!pip install minsearch --break-system-packages

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import minsearch


In [4]:
import json

In [5]:
!wget https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json


--2025-10-03 19:15:32--  https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘documents.json.1’

documents.json.1        [     <=>            ] 858.41K   862KB/s    in 1.0s    

2025-10-03 19:15:34 (862 KB/s) - ‘documents.json.1’ saved [879011]



In [5]:
with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

In [6]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [7]:
documents[0]


{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [8]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

In [9]:
q = 'the course has already started, can I still enroll?'


In [10]:
index.fit(documents)


In [ ]:
import os
from mistralai.client import MistralClient
api_key = "YOUR_API_KEY"
client = MistralClient(api_key=api_key)

response = client.chat(
    model="open-mistral-7b",
    messages=[
        {"role": "user", "content":q}
    ]
)

print(response)

id='6e66b4889df942aeb88650aabed565a2' object='chat.completion' created=1759703308 model='open-mistral-7b' choices=[ChatCompletionResponseChoice(index=0, message=ChatMessage(role='assistant', content="It depends on the specific course and the policy of the institution offering it. Some courses may allow late enrollment, while others may not. It's best to contact the institution directly to inquire about their late enrollment policy. They should be able to tell you if it's still possible to enroll and provide you with the necessary steps to do so.", name=None, tool_calls=None, tool_call_id=None), finish_reason=<FinishReason.stop: 'stop'>)] usage=UsageInfo(prompt_tokens=15, total_tokens=90, completion_tokens=75)


In [12]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [13]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [14]:
def llm(prompt):
    response = client.chat(
    model="open-mistral-7b",
    messages=[
        {"role": "user", "content":prompt}
    ]
)
    return response.choices[0].message.content

In [15]:
query = 'how do I run kafka?'

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [16]:
rag(query)


'To run Kafka in the context of the provided FAQ, you have different options based on the programming language you\'re using:\n\n1. Java: In the project directory, run the following command in your terminal:\n\n```\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```\n\n2. Python: You can create a virtual environment, install the necessary dependencies, and run the Python script in that environment. To create a virtual environment and install the dependencies, follow these steps:\n   - Activate the virtual environment (the path may vary depending on your system):\n\n     ```\n     source env/bin/activate\n     ```\n   - Install the required packages:\n\n     ```\n     pip install -r requirements.txt\n     ```\n   - Run the Python script:\n\n     ```\n     python producer.py\n     ```\n\n3. Python (Docker): Navigate to the /docker/spark directory and execute the following command to fix the permission error:\n\n```\nchmod +x build.sh\n```

In [17]:
rag('the course has already started, can I still enroll?')


"Based on the provided context, you can still enroll in the course even if it has already started. However, you should be aware that there will be deadlines for turning in the final projects. The course will start on the 15th of January, 2024, at 17h00. Before the course starts, you can prepare by installing and setting up all the dependencies and requirements such as a Google cloud account, Google Cloud SDK, Python 3 (installed with Anaconda), Terraform, and Git. You can also look over the prerequisites and syllabus to see if you are comfortable with these subjects. For support during the self-paced mode, you can use the course's Slack channel."

In [18]:
documents[0]


{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [23]:
from elasticsearch import Elasticsearch
es_client = Elasticsearch(
    'http://localhost:9200'
)


In [24]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course-questions"

es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [25]:
documents[0]


{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [26]:
from tqdm.auto import tqdm

In [27]:
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/948 [00:00<?, ?it/s]

In [28]:
query = 'I just disovered the course. Can I still join it?'


In [29]:
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    
    result_docs = []
    
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])
    
    return result_docs

In [30]:
def rag(query):
    search_results = elastic_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [31]:
rag(query)


"Yes, you can still join the course even though it has already started. You're still eligible to submit homeworks, but be aware of the deadlines for turning in the final projects. You can find the course materials and continue learning at your own pace after the course finishes.\n\nBefore the course starts, you can prepare by installing and setting up all the dependencies and requirements such as a Google cloud account, Google Cloud SDK, Python 3 (installed with Anaconda), Terraform, and Git. You can also look over the prerequisites and syllabus to see if you are comfortable with these subjects.\n\nYou can get support if you take the course in the self-paced mode by using the Slack channel. It is recommended to search the channel and the FAQ document first before asking questions. You can also tag the bot @ZoomcampQABot to help you conduct the search."

In [32]:
## RAG with vector search

In [33]:
from qdrant_client import QdrantClient, models

2025-10-06 00:30:57.070067751 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [34]:
qd_client = QdrantClient("http://localhost:6333")

In [35]:
EMBEDDING_DIMENSIONALITY = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"

In [36]:
collection_name = "zoomcamp-faq"

In [37]:
qd_client.delete_collection(collection_name=collection_name)

True

In [38]:
qd_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)

True

In [39]:
qd_client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword"
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [40]:
points = []

for i, doc in enumerate(documents):
    text = doc['question'] + ' ' + doc['text']
    vector = models.Document(text=text, model=model_handle)
    point = models.PointStruct(
        id=i,
        vector=vector,
        payload=doc
    )
    points.append(point)

In [41]:
qd_client.upsert(
    collection_name=collection_name,
    points=points
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [42]:
question = 'I just discovered the course. Can I still join it?'


In [43]:
def vector_search(question):
    print('vector_search is used')
    
    course = 'data-engineering-zoomcamp'
    query_points = qd_client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=question,
            model=model_handle 
        ),
        query_filter=models.Filter( 
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=5,
        with_payload=True
    )
    
    results = []
    
    for point in query_points.points:
        results.append(point.payload)
    
    return results

In [44]:
def rag(query):
    search_results = vector_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [45]:
rag('how do I run kafka?')

vector_search is used


"To run Kafka in the context of your course, follow these steps:\n\n1. Navigate to the project directory where your Java Kafka scripts are located.\n\n2. In the terminal, run the following command to execute a Java script (for example, `JsonProducer.java`):\n\n   ```\n   java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n   ```\n\n   Replace `<jar_name>` with the name of your compiled Java JAR file.\n\n3. Ensure that the `StreamsConfig.BOOTSTRAP_SERVERS_CONFIG` in the scripts is correct, and that the cluster key and secrets are updated in `Secrets.java`.\n\n4. If you encounter the error `kafka.errors.NoBrokersAvailable: NoBrokersAvailable`, check if your Kafka broker Docker container is working. Use `docker ps` to confirm, then in the Docker Compose YAML file folder, run `docker compose up -d` to start all the instances.\n\nIf you are using Python, follow these steps:\n\n1. Create a virtual environment and activate it:\n\n   ```\n   python 